### Завдання 1. Завантаження даних NOAA
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [1]:
import urllib.request
import os
from datetime import datetime

if not os.path.exists('vhi_data'):
    os.makedirs('vhi_data')

def download_vhi_data(province_id, year1=1981, year2=2026):
    existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують. Пропускаємо завантаження.")
        return existing_files[0]

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean"
    
    now = datetime.now()
    date_time_str = now.strftime("%Y%m%d_%H%M%S")
    filename = f"vhi_data/vhi_id_{province_id}_{date_time_str}.csv"
    
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"Успішно завантажено: {filename}")
        return filename
    except Exception as e:
        print(f"Помилка при завантаженні області {province_id}: {e}")
        return None

for i in range(1, 28):
    download_vhi_data(i)

Успішно завантажено: vhi_data/vhi_id_1_20260313_174032.csv
Успішно завантажено: vhi_data/vhi_id_2_20260313_174038.csv
Успішно завантажено: vhi_data/vhi_id_3_20260313_174039.csv
Успішно завантажено: vhi_data/vhi_id_4_20260313_174040.csv
Успішно завантажено: vhi_data/vhi_id_5_20260313_174041.csv
Успішно завантажено: vhi_data/vhi_id_6_20260313_174042.csv
Успішно завантажено: vhi_data/vhi_id_7_20260313_174044.csv
Успішно завантажено: vhi_data/vhi_id_8_20260313_174046.csv
Успішно завантажено: vhi_data/vhi_id_9_20260313_174047.csv
Успішно завантажено: vhi_data/vhi_id_10_20260313_174048.csv
Успішно завантажено: vhi_data/vhi_id_11_20260313_174049.csv
Успішно завантажено: vhi_data/vhi_id_12_20260313_174050.csv
Успішно завантажено: vhi_data/vhi_id_13_20260313_174051.csv
Успішно завантажено: vhi_data/vhi_id_14_20260313_174054.csv
Успішно завантажено: vhi_data/vhi_id_15_20260313_174055.csv
Успішно завантажено: vhi_data/vhi_id_16_20260313_174055.csv
Успішно завантажено: vhi_data/vhi_id_17_20260313_

### Завдання 2. Очищення даних та зміна індексів
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області. Реалізувати процедуру зміни індексів так, щоб області індексувалися за українською абеткою (1 область - Вінницька).

In [2]:
import pandas as pd
import glob
import os

province_mapping = {
    1: (22, 'Черкаська'), 2: (24, 'Чернігівська'), 3: (23, 'Чернівецька'), 
    4: (25, 'АР Крим'), 5: (3, 'Дніпропетровська'), 6: (4, 'Донецька'), 
    7: (8, 'Івано-Франківська'), 8: (19, 'Харківська'), 9: (20, 'Херсонська'), 
    10: (21, 'Хмельницька'), 11: (9, 'Київська'), 12: (26, 'м. Київ'), 
    13: (10, 'Кіровоградська'), 14: (11, 'Луганська'), 15: (12, 'Львівська'), 
    16: (13, 'Миколаївська'), 17: (14, 'Одеська'), 18: (15, 'Полтавська'), 
    19: (16, 'Рівненська'), 20: (27, 'м. Севастополь'), 21: (17, 'Сумська'), 
    22: (18, 'Тернопільська'), 23: (6, 'Закарпатська'), 24: (1, 'Вінницька'), 
    25: (2, 'Волинська'), 26: (7, 'Запорізька'), 27: (5, 'Житомирська')
}

all_files = glob.glob("vhi_data/*.csv")
dfs = []

for file in all_files:
    base_name = os.path.basename(file)
    
    old_id = int(base_name.split('_')[2])
    
    df = pd.read_csv(file, index_col=False, header=None, skiprows=2, skipfooter=1, engine='python',
                     names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Empty'])
    
    df = df.drop('Empty', axis=1, errors='ignore')
    
    df = df[df['VHI'] != -1.0] 
    
    new_id, prov_name = province_mapping[old_id]
    df['Province'] = prov_name
    df['Province_ID'] = new_id
    
    dfs.append(df)

df_clean = pd.concat(dfs, ignore_index=True)

df_clean['Year'] = df_clean['Year'].astype(str).str.replace('<tt><pre>', '')
df_clean['Year'] = df_clean['Year'].astype(int)

df_clean.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province,Province_ID
0,1982,1,0.059,258.24,51.11,48.78,49.95,Хмельницька,21
1,1982,2,0.063,261.53,55.89,38.20,47.04,Хмельницька,21
2,1982,3,0.063,263.45,57.30,32.69,44.99,Хмельницька,21
3,1982,4,0.061,265.10,53.96,28.62,41.29,Хмельницька,21
4,1982,5,0.058,266.42,46.87,28.57,37.72,Хмельницька,21


### Завдання 3. Ряд VHI для області за вказаний рік
Реалізувати процедуру для формування вибірок окремими функціями: ряд VHI для області за вказаний рік.

In [3]:
def get_vhi_by_year_and_province(df, province_id, year):
    filtered_df = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    return filtered_df[['Week', 'VHI', 'Province']]

vhi_2023_vinnytsia = get_vhi_by_year_and_province(df_clean, 1, 2023)
vhi_2023_vinnytsia.head()

,Week,VHI,Province
35802,1,40.09,Вінницька
35803,2,43.61,Вінницька
35804,3,46.74,Вінницька
35805,4,48.03,Вінницька
35806,5,47.88,Вінницька


### Завдання 4. Ряд VHI за вказаний діапазон років
Реалізувати процедуру для формування вибірок окремими функціями: ряд VHI за вказаний діапазон років для вказаних областей.

In [4]:
def get_vhi_by_years_range_and_provinces(df, province_ids, start_year, end_year):
    filtered_df = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    return filtered_df[['Year', 'Week', 'Province', 'VHI']]

vhi_range = get_vhi_by_years_range_and_provinces(df_clean, [1, 9], 2010, 2012)
vhi_range.head(0-30)

,Year,Week,Province,VHI
3654,2010,1,Київська,47.84
3655,2010,2,Київська,47.23
3656,2010,3,Київська,49.70
3657,2010,4,Київська,52.06
3658,2010,5,Київська,52.79
...,...,...,...,...
35247,2012,18,Вінницька,41.64
35248,2012,19,Вінницька,43.38
35249,2012,20,Вінницька,46.84
35250,2012,21,Вінницька,49.49


### Завдання 5. Пошук екстремумів, середнього та медіани
Реалізувати процедуру для формування вибірок окремими функціями: пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [5]:
def get_vhi_statistics(df, province_id, year):
    vhi_data = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]['VHI']
    
    if vhi_data.empty:
        return "Немає даних за цей період."
    
    stats = {
        'Min': vhi_data.min(),
        'Max': vhi_data.max(),
        'Mean': round(vhi_data.mean(), 2),
        'Median': vhi_data.median()
    }
    
    print(f"Статистика VHI для області з ID {province_id} за {year} рік:")
    for key, value in stats.items():
        print(f"{key}: {value}")
        
    return stats

stats_kyiv_2020 = get_vhi_statistics(df_clean, 9, 2020)

Статистика VHI для області з ID 9 за 2020 рік:
Min: 29.27
Max: 56.3
Mean: 41.36
Median: 39.375
